<a href="https://colab.research.google.com/github/kdrhanegrilmez/mL-Notebooks/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📥 Adım 1: Veri Setini İndir ve Yükle (Titanic)
Yeni bir hücre aç (Hücrenin üzerine gelip + Kod butonuna bas) ve bunu yapıştır:

In [ ]:
# Colab'in ilk hücresi: Kütüphaneleri içe aktarıyoruz.
# Veri analizi ve matris işlemleri için Pandas ve Numpy:
import numpy as np
import pandas as pd

# Modelleme ve ön işleme için Sklearn araçları:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Veri setini internetten indirmek için (Colab'deki bir özellik):
import urllib.request


In [ ]:
# Titanic veri setini direkt internetten indirip Python ortamına yüklüyoruz.
# Kolaylık olsun diye GitHub'dan bir link kullanıyorum.
url = 'https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/train.csv'

# Pandas'ın read_csv fonksiyonu ile veriyi oku ve bir "DataFrame" (tablo) haline getir.
df=pd.read_csv(url)


# Tablonun ilk 5 satırını ekrana yazdır:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


👨‍🏫 Mentor Notu: df.head() komutu sana veri setini gösterir. PassengerId, Survived (bizim tahmin etmek istediğimiz etiket), Age, Sex, Fare gibi sütunları göreceksin. Senin burada yapman gereken, tablodaki veri tiplerini (Sayı mı? Metin mi?) incelemek


Adım 2: Veri Temizleme, Kayıp Veri ve Kodlama (Ön İşleme)
Yeni bir hücre aç ve bunu çalıştır. Bu adım en önemlisi! Eğer modeli eğitmeden önce bu adımları atlarsan modelin çalışmaz veya saçma sonuçlar verir.

In [ ]:
# 1. İşe yaramayan sütunları silelim. (PassengerId, Name, Ticket, Cabin - Çünkü KNN mesafeyi burada hesaplayamaz veya çok gereksiz bilgi).
df=df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1)

# 2. Kayıp Veri Problemi (Missing Data):
# Age sütununda eksik değerler var. Bunları yaş ortalaması ile dolduralım.

df['Age'] = df['Age'].fillna(df['Age'].median())
# Embarked (Biniş Limanı) sütununda eksikler var. Bunu en sık kullanılan liman ile dolduralım.
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3. Kategorik Veriyi Kodlama (Encoding):
# KNN mesafe hesaplar (Öklid,Minkowski,Manhatten vs.). "Kadın" ve "Erkek" metinleriyle işlem yapamaz.
# Bu yüzden Cinsiyet (Sex) ve Liman (Embarked) sütunlarını sayısal değerlere (0 ve 1) çeviriyoruz.
# drop_first=True demek, "Kadın mı? Evet/Hayır" gibi binary (ikili) bir sütun oluştur demektir.

df = pd.get_dummies(df, columns=['Sex','Embarked'],drop_first=True)
# Son halini görelim:
df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


In [ ]:
# Hedef değişkenimiz (tahmin etmek istediğimiz) 'Survived' sütunu. Buna "Y" diyoruz.
y=df['Survived']
# Özniteliklerimiz (girdi) ise geri kalan tüm sütunlar. Buna "X" diyoruz.
x=df.drop('Survived',axis=1)
# Veriyi %80 Eğitim, %20 Test olarak ayırıyoruz.
# random_state=42 demek, her çalıştırdığımızda aynı rastgele bölünmeyi al demek (tekrarlanabilirlik için).
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

print(f"Eğitim verisi boyutu: {x_train.shape}")
print(f"Test verisi boyutu: {x_test.shape}")



Eğitim verisi boyutu: (712, 8)
Test verisi boyutu: (179, 8)


👨‍🏫 Mentor Notu: Makine öğrenmesinde, modelin elinde sadece X_train ve y_train olacak. X_test ve y_test'i hiç görmeyecek. Biz modeli eğitip, sonra X_test'i verip tahmin yapmasını isteyeceğiz. Bu işleme "Genelleme (Generalization)" denir.

In [ ]:
# Pipeline, bir işçi zinciridir. İş akışını tek bir pakette toplar.
# Burada diyoruz ki:
# 1. Önce veriyi Standardize et (StandartScaler)
# 2. Sonra KNN modelini uygula (KNeighborsClassifier)

pipeline= Pipeline(
    [
        ('scaler',StandardScaler()),
        ('knn',KNeighborsClassifier())
    ]
)

# Hiperparametre Optimizasyonu (GridSearchCV):
# K değerini (n_neighbors) 1'den 30'a kadar tek tek dene ve hangisinin en iyi olduğunu bul.

param_grid={'knn__n_neighbors':range(1,31)}

# 5-Katlı Çapraz Doğrulama ile en iyi K değerini seç.
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(x_train, y_train)

print("BULUNAN EN İYİ K DEĞERİ:", grid_search.best_params_)
print("EN İYİ EĞİTİM BAŞARISI:", grid_search.best_score_)





BULUNAN EN İYİ K DEĞERİ: {'knn__n_neighbors': 13}
EN İYİ EĞİTİM BAŞARISI: 0.8230473751600511


👨‍🏫 Mentor Notu: Burada neden StandardScaler kullandık? Çünkü Age (0-80 arası) ve Fare (0-500 arası) çok farklı ölçeklerde. KNN mesafe hesapladığı için, eğer ölçeklendirmezsen, bilet fiyatı yaştan çok daha ağır basar ve model saçmalar.

In [ ]:
# Test verisi üzerinde tahmin yap (Modelin hiç görmediği veri)
y_predict=grid_search.predict(x_test)
print("Doğruluk (Accuracy) Skoru:", accuracy_score(y_test, y_predict))
print("\nKarmaşıklık Matrisi (Confusion Matrix):")
print(confusion_matrix(y_test,y_predict))
print("\nSınıflandırma Raporu:")
print(classification_report(y_test,y_predict))


Doğruluk (Accuracy) Skoru: 0.8044692737430168

Karmaşıklık Matrisi (Confusion Matrix):
[[93 12]
 [23 51]]

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       105
           1       0.81      0.69      0.74        74

    accuracy                           0.80       179
   macro avg       0.81      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179

